# 04 — Fold-5 comparison-sample review

**Goal:** show whether the images the audit flags differ from typical images. The paper's manual review looked at the
20 highest artifact-scored training images of fold 5. This notebook adds two comparison groups of equal size from the
same fold, the 20 lowest-scored and 20 random images, and all 60 are rated **blind**.

**What it does**
1. Re-runs Loop 1 for fold 5 only (same seed and code as `01_exclusion.ipynb`; roughly 30-60 min on a Colab GPU) and the same audit.
2. Selects the top-20, lowest-20 and random-20 images by artifact score.
3. Saves the 60 Grad-CAM overlays under random names (`IMG_000.png` ...) plus a blank rating sheet.
4. After the sheet is filled in, counts burn-in and centered attention per group and runs Fisher exact tests.

**How to run**
* Runtime -> Change runtime type -> GPU, then Run all. Put the dataset where `DATASET_ROOT` (Cell 2) points (see the repo README).
* Cell 1 prints the software versions used (also saved to `versions.json`).
* Cell 11 prints a check of Loop 1 Macro-F1 for fold 5 against `EXPECTED_L1_F1` (0.4074, the fold-5 value from `01_exclusion.ipynb`). A small difference can occur on a different GPU type or library build; if it differs, state that the fold-5 audit was re-run.
* Rate each overlay with two yes/no criteria: **Is Grad-CAM attention anatomically centered? Is an on-screen recording artifact (timestamp/label burn-in) visible?**
* **Blinding:** ideally the rater did not run the notebook and does not open `review_key_DO_NOT_OPEN_BEFORE_RATING.csv` before rating.
* Results are written to `RESULTS_DIR` on the Colab VM (`/content/comparison_review_fold5`), which is lost when the session ends. Download the folder after Cell 11, fill in `rating_sheet.csv`, upload it back to the same path, then run Cell 12.


In [ ]:
# CELL 1: INSTALL + IMPORTS + VERSIONS
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "scipy"])

import os, json, random, re, warnings, gc, platform
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
from PIL import Image
import sklearn, scipy
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score
from scipy import stats

warnings.filterwarnings('ignore')

VERSIONS = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "scikit-learn": sklearn.__version__,
    "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
}
print("Software versions (paste into Sec. II.G of the paper):")
for k, v in VERSIONS.items():
    print(f"  {k:13s} {v}")

In [ ]:
# CELL 2: CONFIGURATION (identical to 01_exclusion.ipynb, plus the review settings at the bottom)
# Malhari dataset from Mendeley Data (DOI 10.17632/m5kxdj7m36.1); see the repo README.
DATASET_ROOT = "/content/data/Malhari_Dataset_V1/colposcopic"
AUG_ROOT     = "/content/mendeley_augmented"
OUTPUT_DIR   = "/content/outputs"                                   # temporary model checkpoints
RESULTS_DIR  = "/content/comparison_review_fold5"                   # on the Colab VM: download before the session ends

SEED            = 42
BATCH_SIZE      = 8
IMG_SIZE        = 224
NUM_WORKERS     = 2
MAX_EPOCHS_P1   = 15
MAX_EPOCHS_P2   = 30
LR_P1           = 1e-3
LR_P2           = 1e-4
LABEL_SMOOTHING = 0.15
WEIGHT_DECAY    = 1e-4
DROPOUT         = 0.3
MIXUP_ALPHA     = 0.2
PATIENCE        = 8
PRETRAINED      = True

ENSEMBLE_ARCHS       = ["resnet18", "mobilenetv3", "efficientnet_b0"]
DST_UNCERTAINTY_MASS = 0.1
K_THRESHOLD          = 0.3
EXCLUDE_PERCENTILE   = 95      # top 5% flagged
RELABEL_MIN_AGREE    = 2
RELABEL_MIN_CONF     = 0.6
W_DST, W_XAI         = 0.5, 0.5  # combined audit score weights (main analysis)

# ── Comparison review settings ────────────────────────────────────────────
N_FOLDS       = 5
FOLD_TO_RUN   = 4        # 0-based index → fold 5, the fold used for the paper's figures and top-20 review
REVIEW_N      = 20       # images per group (top / lowest / random)
EXPECTED_L1_F1 = 0.4074  # fold-5 Loop 1 Macro-F1 from 01_exclusion.ipynb
ORIGINAL_TOP20_BURN_IN = 9   # 9/20 reported in the paper

CLASS_ORDER     = ["CIN1", "CIN2", "CIN3"]
FOLDER_TO_LABEL = {"CIN1": "CIN1", "CIN2": "CIN2", "CIN3": "CIN3"}
NUM_CLASSES  = len(CLASS_ORDER)
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_LABEL = {i: c for c, i in LABEL_TO_IDX.items()}

def set_seed(s=SEED):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    np.random.seed(s); random.seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(f"{RESULTS_DIR}/versions.json", "w") as f:
    json.dump(VERSIONS, f, indent=2)
print(f"Device: {DEVICE} | Results → {RESULTS_DIR}")

In [ ]:
# CELL 3: 10× AUGMENTED COPIES (identical to the augmentation cell of 01_exclusion.ipynb; skipped if already generated)
from PIL import ImageEnhance
COPIES_PER_IMAGE = 10

def random_augment(img):
    ops = [
        lambda x: transforms.RandomHorizontalFlip(1.0)(x),
        lambda x: transforms.RandomVerticalFlip(1.0)(x),
        lambda x: transforms.RandomRotation(30)(x),
        lambda x: transforms.RandomAffine(0, translate=(0.15, 0.15))(x),
        lambda x: transforms.RandomAffine(0, scale=(0.8, 1.2))(x),
        lambda x: transforms.RandomAffine(0, shear=15)(x),
        lambda x: transforms.RandomPerspective(distortion_scale=0.2, p=1.0)(x),
        lambda x: transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))(x),
        lambda x: ImageEnhance.Brightness(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Contrast(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Sharpness(x).enhance(random.uniform(0.5, 2.0)),
    ]
    for op in random.sample(ops, random.randint(3, 5)):
        img = op(img)
    return img

def n_files(root):
    return sum(len(fs) for _, _, fs in os.walk(root)) if os.path.isdir(root) else 0

if n_files(AUG_ROOT) == 0:
    total = 0
    for cin_class in ["CIN1", "CIN2", "CIN3"]:
        class_path = os.path.join(DATASET_ROOT, cin_class)
        for patient in sorted(os.listdir(class_path)):
            patient_path = os.path.join(class_path, patient)
            if not os.path.isdir(patient_path):
                continue
            out_dir = os.path.join(AUG_ROOT, cin_class, patient)
            os.makedirs(out_dir, exist_ok=True)
            for fname in sorted(os.listdir(patient_path)):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                img = Image.open(os.path.join(patient_path, fname)).convert("RGB")
                stem = fname.rsplit('.', 1)[0]
                img.save(os.path.join(out_dir, fname))
                for k in range(COPIES_PER_IMAGE):
                    random.seed(total * 1000 + k); np.random.seed(total * 1000 + k)
                    torch.manual_seed(total * 1000 + k)
                    random_augment(img.copy()).save(os.path.join(out_dir, f"{stem}_{k}.jpg"))
                total += 1
    print(f"Generated augmented copies for {total} originals.")
else:
    print(f"Augmented folder already present ({n_files(AUG_ROOT)} files): skipping generation.")
set_seed()

In [ ]:
# CELL 4: DATASET DISCOVERY (same as 01_exclusion.ipynb, plus a `source_key` linking every augmented copy to its original)
def discover_mendeley(root):
    records = []
    root = Path(root)
    for cin_class in sorted(os.listdir(root)):
        class_path = root / cin_class
        if not class_path.is_dir() or cin_class not in FOLDER_TO_LABEL:
            continue
        label = FOLDER_TO_LABEL[cin_class]
        for patient_folder in sorted(os.listdir(class_path)):
            patient_path = class_path / patient_folder
            if not patient_path.is_dir():
                continue
            for fname in sorted(os.listdir(patient_path)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    records.append({
                        "filepath": str(patient_path / fname), "filename": fname,
                        "folder": cin_class,
                        "patient_id": f"{cin_class}_{patient_folder.strip()}",
                        "patient_folder": patient_folder,
                        "label": label, "label_idx": LABEL_TO_IDX[label],
                    })
    return pd.DataFrame(records)

df = discover_mendeley(DATASET_ROOT)
df_aug = discover_mendeley(AUG_ROOT)

# Link each original and each augmented file to a source key (folder, patient_folder, original stem)
df["source_key"] = [f"{r.folder}|{r.patient_folder}|{r.filename.rsplit('.', 1)[0]}" for r in df.itertuples()]
orig_keys = set(df["source_key"])

def aug_source(r):
    stem = r.filename.rsplit('.', 1)[0]
    key = f"{r.folder}|{r.patient_folder}|{stem}"
    if key in orig_keys:                       # identical copy of the original
        return key
    key2 = f"{r.folder}|{r.patient_folder}|{re.sub(r'_[0-9]+$', '', stem)}"
    return key2 if key2 in orig_keys else None

df_aug["source_key"] = [aug_source(r) for r in df_aug.itertuples()]
n_unmatched = df_aug["source_key"].isna().sum()
print(f"Originals: {len(df)} images, {df['patient_id'].nunique()} patients | "
      f"Augmented pool: {len(df_aug)} | unmatched augmented files: {n_unmatched}")
assert n_unmatched == 0, "Some augmented files could not be linked to an original: check file names."
copies = df_aug.groupby("source_key").size()
print(f"Files per original in the augmented folder: min={copies.min()}, max={copies.max()} (expected 11)")

In [ ]:
# CELL 5: TRANSFORMS + DATASET + LOADERS
# Only change: an optional `sample_weight` column multiplies the class-balancing sampler weights.
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ColposcopyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label_idx"], row["filepath"]

def make_loaders(tr_df, v_df, te_df):
    tr_ds = ColposcopyDataset(tr_df, eval_transforms)
    v_ds  = ColposcopyDataset(v_df,  eval_transforms)
    te_ds = ColposcopyDataset(te_df, eval_transforms)
    cw = {}
    c = tr_df["label_idx"].value_counts().sort_index()
    for idx in range(NUM_CLASSES):
        cw[idx] = np.sqrt(len(tr_df) / (NUM_CLASSES * c.get(idx, 1)))
    mw = np.mean(list(cw.values()))
    cw = {k: v / mw for k, v in cw.items()}
    sample_w = np.array([cw[idx] for idx in tr_df["label_idx"]], dtype=float)
    if "sample_weight" in tr_df.columns:                       # ← NEW: down-weighting hook
        sample_w = sample_w * tr_df["sample_weight"].to_numpy(dtype=float)
    sampler = WeightedRandomSampler(sample_w.tolist(), len(tr_df), replacement=True)
    g = torch.Generator(); g.manual_seed(SEED)
    pin = torch.cuda.is_available()
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                           pin_memory=pin, generator=g, drop_last=True)
    v_loader  = DataLoader(v_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    return tr_loader, v_loader, te_loader, cw

In [ ]:
# CELL 6: PATIENT-LEVEL GROUP K-FOLD (identical)
gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(df, df["label_idx"], groups=df["patient_id"]))
for i, (tr, te) in enumerate(folds):
    leak = set(df.iloc[tr]["patient_id"]) & set(df.iloc[te]["patient_id"])
    print(f"  Fold {i+1}: train={len(tr)} test={len(te)} | patient leakage={len(leak)}")

In [ ]:
# CELL 7: MODEL (identical)
GRADCAM_LAYERS = {"resnet18": "backbone.layer4", "mobilenetv3": "backbone.features.12",
                  "efficientnet_b0": "backbone.features.8"}

class CINClassifier(nn.Module):
    def __init__(self, arch_name="resnet18", num_classes=NUM_CLASSES, dropout=DROPOUT, pretrained=PRETRAINED):
        super().__init__()
        self.arch_name = arch_name
        if arch_name == "resnet18":
            base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
            self.feat_dim = base.fc.in_features; base.fc = nn.Identity()
        elif arch_name == "mobilenetv3":
            base = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None)
            self.feat_dim = base.classifier[0].in_features; base.classifier = nn.Identity()
        elif arch_name == "efficientnet_b0":
            base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
            self.feat_dim = base.classifier[1].in_features; base.classifier = nn.Identity()
        self.backbone = base
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.feat_dim, num_classes))
    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats), feats
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
    def get_gradcam_target(self):
        layer = self
        for p in GRADCAM_LAYERS[self.arch_name].split("."):
            layer = layer[int(p)] if p.isdigit() else getattr(layer, p)
        return layer

In [ ]:
# CELL 8: TRAINING FUNCTIONS (identical)
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total = 0.0, 0
    all_p, all_l = [], []
    for images, labels, _ in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        mixed, y_a, y_b, lam = mixup_data(images, labels)
        logits, _ = model(mixed)
        loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
        all_p.extend(logits.argmax(1).cpu().numpy()); all_l.extend(labels.cpu().numpy())
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0); total += labels.size(0)
    return total_loss / total, f1_score(all_l, all_p, average='macro', zero_division=0)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total = 0.0, 0
    all_l, all_prob = [], []
    for images, labels, _ in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits, _ = model(images)
        total_loss += criterion(logits, labels).item() * labels.size(0); total += labels.size(0)
        all_l.extend(labels.cpu().numpy()); all_prob.extend(torch.softmax(logits, 1).cpu().numpy())
    return total_loss / total, np.array(all_l), np.array(all_prob)

def train_model(model, tr_loader, v_loader, tag, cw):
    save_dir = f"{OUTPUT_DIR}/{tag}"; os.makedirs(save_dir, exist_ok=True)
    wt = torch.tensor([cw[i] for i in range(NUM_CLASSES)], dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=wt, label_smoothing=LABEL_SMOOTHING)
    best_vl, patience_ctr = float('inf'), 0
    for phase, max_ep, lr, eta in [(1, MAX_EPOCHS_P1, LR_P1, 1e-6), (2, MAX_EPOCHS_P2, LR_P2, 1e-7)]:
        if phase == 1:
            model.freeze_backbone()
            opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=WEIGHT_DECAY)
        else:
            model.load_state_dict(torch.load(f"{save_dir}/best.pt", map_location=DEVICE, weights_only=True))
            model.unfreeze_backbone()
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
            patience_ctr = 0
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_ep, eta_min=eta)
        for ep in range(1, max_ep + 1):
            train_one_epoch(model, tr_loader, criterion, opt)
            vl, _, _ = evaluate(model, v_loader, criterion)
            sched.step()
            if vl < best_vl:
                best_vl, patience_ctr = vl, 0
                torch.save(model.state_dict(), f"{save_dir}/best.pt")
            else:
                patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break
    model.load_state_dict(torch.load(f"{save_dir}/best.pt", map_location=DEVICE, weights_only=True))
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

def train_and_test_ensemble(tr_df, v_df, te_df, tag):
    """Train the 3-model ensemble on tr_df; return test Macro-F1, accuracy, models."""
    tr_loader, v_loader, te_loader, cw = make_loaders(tr_df, v_df, te_df)
    ens = {name: CINClassifier(arch_name=name).to(DEVICE) for name in ENSEMBLE_ARCHS}   # create all, then train (same RNG order as original)
    for name, model in ens.items():
        train_model(model, tr_loader, v_loader, f"{tag}_{name}", cw)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    probs = []
    for m in ens.values():
        _, labels, p = evaluate(m, te_loader, crit); probs.append(p)
    preds = np.mean(probs, axis=0).argmax(1)
    return (float(f1_score(labels, preds, average='macro', zero_division=0)),
            float(accuracy_score(labels, preds)), ens)

In [ ]:
# CELL 9: DST + GRAD-CAM HELPERS (identical to the pipeline cell of 01_exclusion.ipynb)
def softmax_to_mass(probs, unc=DST_UNCERTAINTY_MASS):
    mass = {frozenset({i}): p * (1.0 - unc) for i, p in enumerate(probs)}
    mass[frozenset(range(len(probs)))] = unc
    return mass

def dempster_combine(m1, m2):
    combined, K = {}, 0.0
    for A, mA in m1.items():
        for B, mB in m2.items():
            inter = A & B
            if len(inter) == 0: K += mA * mB
            else: combined[inter] = combined.get(inter, 0.0) + mA * mB
    if K >= 1.0:
        return {frozenset({i}): 1.0 / NUM_CLASSES for i in range(NUM_CLASSES)}, K
    return {k: v / (1.0 - K) for k, v in combined.items()}, K

def combine_multiple(mass_list):
    result, total_K = mass_list[0], 0.0
    for i in range(1, len(mass_list)):
        result, k = dempster_combine(result, mass_list[i])
        total_K = 1 - (1 - total_K) * (1 - k)
    return result, total_K

def belief(mass):
    return {i: sum(m for A, m in mass.items() if A.issubset(frozenset({i}))) for i in range(NUM_CLASSES)}

class GradCAM:
    def __init__(self, model):
        self.model = model
        target = model.get_gradcam_target()
        self.h1 = target.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self.h2 = target.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
    def remove(self):
        self.h1.remove(); self.h2.remove()
    def generate(self, image):
        self.model.eval()
        image = image.to(DEVICE).requires_grad_(True)
        logits, _ = self.model(image)
        pred = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, pred].backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min()) if cam.max() - cam.min() > 1e-8 else np.zeros_like(cam)
        return cam, pred

def peripheral_score(hm):
    h, w = hm.shape; b = int(min(h, w) * 0.2)
    mask = np.zeros_like(hm, dtype=bool)
    mask[:b, :] = mask[-b:, :] = mask[:, :b] = mask[:, -b:] = True
    return float(hm[mask].sum() / hm.sum()) if hm.sum() > 1e-8 else 0.0

def uniformity_score(hm):
    if hm.max() < 1e-8: return 1.0
    h = (hm / (hm.sum() + 1e-8)).flatten(); h = h[h > 1e-10]
    return float(-np.sum(h * np.log(h)) / np.log(hm.size))

def color_channel_score(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = (img_tensor.cpu() * std + mean).clamp(0, 1)
    ch = img.mean(dim=(1, 2)).numpy()
    if ch.sum() < 1e-8: return 0.0, "none"
    fr = ch / ch.sum()
    return float((fr.max() - 1 / 3) / (2 / 3)), ["Red", "Green", "Blue"][fr.argmax()]

def expand_with_augmented(fold_train_orig, df_aug):
    """Identical to 01_exclusion.ipynb (keeps Loop 1 reproducible): pulls all augmented-folder files of the training patients."""
    pts = set(zip(fold_train_orig["folder"], fold_train_orig["patient_folder"]))
    aug = df_aug[[(f, p) in pts for f, p in zip(df_aug["folder"], df_aug["patient_folder"])]]
    aug = aug[~aug["filepath"].isin(set(fold_train_orig["filepath"]))]
    return pd.concat([fold_train_orig, aug], ignore_index=True)

In [ ]:
# CELL 10: AUDIT (same scoring as 01_exclusion.ipynb) + OVERLAY HELPER
def run_audit(ens, fold_train):
    """DST + Grad-CAM on this fold's TRAINING originals only (no validation/test images)."""
    loader = DataLoader(ColposcopyDataset(fold_train, eval_transforms), batch_size=32, shuffle=False, num_workers=NUM_WORKERS)
    names = list(ens.keys())
    for m in ens.values(): m.eval()
    rows = []
    with torch.no_grad():
        for images, labs, fps in loader:
            probs = {a: torch.softmax(ens[a](images.to(DEVICE))[0], 1).cpu().numpy() for a in names}
            for i in range(images.size(0)):
                comb, K = combine_multiple([softmax_to_mass(probs[a][i]) for a in names])
                bel = belief(comb); dst_pred = max(bel, key=bel.get); true = int(labs[i])
                preds = [int(probs[a][i].argmax()) for a in names]
                confs = [float(probs[a][i].max()) for a in names]
                maj, cnt = Counter(preds).most_common(1)[0]
                agree_conf = np.mean([c for p, c in zip(preds, confs) if p == maj])
                high_k, correct = K > K_THRESHOLD, dst_pred == true
                quad = ("MISLABEL_CANDIDATE" if high_k and not correct else "AMBIGUOUS_CASE" if high_k
                        else "MODEL_FAILURE" if not correct else "CLEAN")
                rows.append({"filepath": fps[i], "true_label": IDX_TO_LABEL[true], "conflict_K": K, "quadrant": quad,
                             "relabel_candidate": bool(maj != true and cnt >= RELABEL_MIN_AGREE
                                                       and K < K_THRESHOLD and agree_conf >= RELABEL_MIN_CONF),
                             **{f"pred_{a}": IDX_TO_LABEL[p] for a, p in zip(names, preds)},
                             **{f"conf_{a}": round(c, 4) for a, c in zip(names, confs)}})
    audit = pd.DataFrame(rows)
    gc_ = GradCAM(ens[names[0]])          # ResNet18, as in the paper
    xai = {}
    for images, labs, fps in loader:
        for i in range(images.size(0)):
            hm, _ = gc_.generate(images[i].unsqueeze(0))
            p, u = peripheral_score(hm), uniformity_score(hm)
            c, d = color_channel_score(images[i])
            xai[fps[i]] = {"artifact_score": 0.35 * p + 0.30 * u + 0.35 * c,
                           "peripheral": p, "uniformity": u, "color_channel": c, "dominant_channel": d}
    gc_.remove()
    audit = audit.join(pd.DataFrame.from_dict(xai, orient="index"), on="filepath")
    audit["combined_score"] = W_DST * audit["conflict_K"] + W_XAI * audit["artifact_score"]
    thr = np.percentile(audit["combined_score"], EXCLUDE_PERCENTILE)
    audit["flagged"] = audit["combined_score"] >= thr
    audit = audit.merge(fold_train[["filepath", "source_key", "patient_id"]], on="filepath", how="left")
    return audit

def save_overlay(model, filepath, out_png):
    gc_ = GradCAM(model)
    x = eval_transforms(Image.open(filepath).convert("RGB")).unsqueeze(0)
    hm, _ = gc_.generate(x); gc_.remove()
    img = np.array(Image.open(filepath).convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(img); axs[0].set_title("Original")
    axs[1].imshow(hm, cmap='jet'); axs[1].set_title("Grad-CAM")
    axs[2].imshow(img * 0.5 + plt.cm.jet(hm)[:, :, :3] * 0.5); axs[2].set_title("Overlay")
    for a in axs: a.axis("off")
    plt.tight_layout(); plt.savefig(out_png, dpi=100); plt.close(fig)

In [ ]:
# CELL 11: FOLD 5 → LOOP 1 → AUDIT → BLINDED REVIEW PACKAGE (skipped if already done)
key_csv = f"{RESULTS_DIR}/review_key_DO_NOT_OPEN_BEFORE_RATING.csv"
sheet_csv = f"{RESULTS_DIR}/rating_sheet.csv"
if os.path.exists(key_csv):
    print("Review package already exists: go to Cell 12.")
else:
    fold_num = FOLD_TO_RUN
    train_idx, test_idx = folds[fold_num]
    set_seed(SEED + fold_num)
    fold_trainval = df.iloc[train_idx].reset_index(drop=True)
    fold_test = df.iloc[test_idx].reset_index(drop=True)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    tv_tr, tv_va = next(gss.split(fold_trainval, fold_trainval["label_idx"], groups=fold_trainval["patient_id"]))
    fold_train = fold_trainval.iloc[tv_tr].reset_index(drop=True)
    fold_val = fold_trainval.iloc[tv_va].reset_index(drop=True)
    assert not (set(fold_train.patient_id) & (set(fold_test.patient_id) | set(fold_val.patient_id)))
    base_expanded = expand_with_augmented(fold_train, df_aug)
    print(f"Fold {fold_num+1}: train originals {len(fold_train)} (expanded {len(base_expanded)}) | "
          f"val {len(fold_val)} | test {len(fold_test)}")

    f1, acc, ens = train_and_test_ensemble(base_expanded, fold_val, fold_test, "L1")
    match = "matches" if abs(f1 - EXPECTED_L1_F1) < 1e-3 else "DIFFERS from"
    print(f"★ Loop 1 Macro-F1 = {f1:.4f} ({match} the expected {EXPECTED_L1_F1:.4f}) | Acc = {acc:.3f}")

    audit = run_audit(ens, fold_train)
    audit.to_csv(f"{RESULTS_DIR}/audit_fold5.csv", index=False)
    print(f"Audited {len(audit)} training images | r(K, artifact) = "
          f"{np.corrcoef(audit.conflict_K, audit.artifact_score)[0,1]:+.3f} (paper: −0.09)")

    # Groups by artifact score, as in the original top-20 review
    a = audit.sort_values("artifact_score", ascending=False).reset_index(drop=True)
    top, low = a.head(REVIEW_N), a.tail(REVIEW_N)
    middle = a.iloc[REVIEW_N:len(a) - REVIEW_N]
    rnd = middle.sample(n=REVIEW_N, random_state=SEED)
    groups = pd.concat([top.assign(group="top20"), low.assign(group="lowest20"), rnd.assign(group="random20")],
                       ignore_index=True)

    # Blinded overlays: random IDs, no scores or group in the image
    os.makedirs(f"{RESULTS_DIR}/blinded_overlays", exist_ok=True)
    perm = np.random.default_rng(SEED + 999).permutation(len(groups))
    groups["blind_id"] = [f"IMG_{i:03d}" for i in perm]
    for r in groups.itertuples():
        save_overlay(ens[ENSEMBLE_ARCHS[0]], r.filepath, f"{RESULTS_DIR}/blinded_overlays/{r.blind_id}.png")
    groups[["blind_id", "group", "filepath", "true_label", "artifact_score", "conflict_K", "combined_score",
            "flagged", "peripheral", "uniformity", "color_channel"]].to_csv(key_csv, index=False)
    pd.DataFrame({"blind_id": sorted(groups["blind_id"]), "attention_centered_Y_N": "",
                  "burn_in_artifact_Y_N": "", "notes": ""}).to_csv(sheet_csv, index=False)
    json.dump({"loop1_macro_f1": f1, "loop1_acc": acc, "n_audited": len(audit), **VERSIONS},
              open(f"{RESULTS_DIR}/run_info.json", "w"), indent=2)
    print(f"\n{len(groups)} blinded overlays → {RESULTS_DIR}/blinded_overlays/")
    print(f"Fill in {sheet_csv} (Y/N per image), save it to the same folder, then run Cell 12.")

In [ ]:

# CELL 12: SCORE THE REVIEW (run after rating_sheet.csv is filled in)
key = pd.read_csv(key_csv)
sheet = pd.read_csv(sheet_csv, dtype=str).fillna("")
if (sheet["burn_in_artifact_Y_N"].str.strip() == "").any() or (sheet["attention_centered_Y_N"].str.strip() == "").any():
    print("rating_sheet.csv is not completely filled in yet.")
else:
    m = key.merge(sheet, on="blind_id")
    for col in ["attention_centered_Y_N", "burn_in_artifact_Y_N"]:
        m[col] = m[col].str.strip().str.upper().str.startswith("Y")
    order = ["top20", "lowest20", "random20"]
    tab = (m.groupby("group").agg(n=("blind_id", "size"), burn_in=("burn_in_artifact_Y_N", "sum"),
                                  centered=("attention_centered_Y_N", "sum")).reindex(order).astype(int))
    print(tab, "\n")
    t = tab.loc["top20"]
    tests = {}
    for o in ["lowest20", "random20"]:
        b = tab.loc[o]
        tests[o] = stats.fisher_exact([[t.burn_in, t.n - t.burn_in], [b.burn_in, b.n - b.burn_in]])[1]
        print(f"Burn-in: top-20 {t.burn_in}/{t.n} vs {o} {b.burn_in}/{b.n} → Fisher exact p = {tests[o]:.3f}")
    print(f"\n(Blinded top-20 burn-in {t.burn_in}/{t.n} vs. the paper's original unblinded {ORIGINAL_TOP20_BURN_IN}/20.)")
    tab.to_csv(f"{RESULTS_DIR}/review_results.csv")
    lo, ra = tab.loc["lowest20"], tab.loc["random20"]
    print("\nDraft sentence for Sec. III.B (check it against the numbers):")
    print(f"In a blinded comparison-sample review of fold 5, on-screen recording artifacts were present in "
          f"{t.burn_in}/{t.n} of the highest artifact-scored images, versus {lo.burn_in}/{lo.n} of the lowest-scored and "
          f"{ra.burn_in}/{ra.n} of randomly selected images (Fisher exact p = {tests['lowest20']:.3f} and "
          f"{tests['random20']:.3f}, respectively); Grad-CAM attention was anatomically centered in "
          f"{t.centered}/{t.n}, {lo.centered}/{lo.n}, and {ra.centered}/{ra.n} images.")